Self-Attention数学原理

In [ ]:
import torch

# 考虑下面这个玩具示例（简化版，方便理解原理）

torch.manual_seed(1337)
# 固定随机种子，保证每次运行随机数都一样（方便复现结果）

B, T, C = 4, 8, 2
# B=4: 4个样本（batch）
# T=8: 每个样本8个时间步（序列长度8）
# C=2: 每个时间步用2维向量表示（这里C不是vocab_size了！）

x = torch.randn(B, T, C)
# 随机生成一个形状(4,8,2)的张量
# 模拟"已经经过embedding"的输入数据

x.shape
# torch.Size([4, 8, 2])

torch.Size([4, 8, 2])

用两个 ⁠for⁠ 循环求平均实现因果遮蔽

note：

假设输入的数据尺寸是 ⁠(B, T, C)⁠：


 B (Batch Size)：假设 = 1（一共就 1 句话）


 T (Time/Sequence Length)：序列长度 = 4（比如这句话是 ⁠[我, 喜, 欢, 猫]⁠）


 C (Channels/Embedding Dim)：词向量维度

 在之前的二元语法（Bigram）模型中，我们预测下一个字符时，只看当前这一个字符，完全不管过去更早的历史 。比如输入 "I love AI"，当我们在 "AI" 这个位置时，模型只拿着 "AI" 去猜，完全忘记了前面还出现过 "I" 和 "love"。

我们希望：当我们在时间步
t
 时，模型能够把从
0
 到
t
 的所有历史字符特征融合起来。
最简单的融合方法：就是把它们全部加起来求平均。
如果我们在第
t
 个位置，我们就把前面
t
+
1
 个字符的特征向量全部加起来除以
t
+
1
，得到一个综合了历史信息的平均特征向量，作为当前位置的“历史记忆表示”

In [ ]:
# B: Batch_size (4个句子)
# T: Time/Sequence Length (8个字符)
# C: Channel/Embedding Dimension (65维，每个字符的特征维度)

# 1. 初始化一个全零的 3D 张量，作为记忆'词袋'
xbow = torch.zeros((B, T, C))

# 外层循环：遍历每一个句子 b (Batch 维度)
for b in range(B):
    # 内层循环：遍历每一个时间步 t (Time 维度)
    for t in range(T):
        # 切片：取出当前句子 b 中，从第 0 个字符到当前第 t 个字符的所有历史数据
        # 此时 xprev 的形状为: (t+1, C) ──> 代表包含当前在内，过去一共有 t+1 个字符
        xprev = x[b, :t+1]

        # 计算：在第 0 维（时间轴）上求平均值
        # 这一步将 (t+1, C) 的多行历史数据压缩，融合成了一个单一的 C 维平均向量
        # 并将其写入我们“记忆”对应的 xbow[b, t] 位置中
        xbow[b, t] = torch.mean(xprev, 0)

In [ ]:
xbow

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

B（最外层，有 4 个大格子）：
Batch_size = 4（代表有 4 个完全独立的句子）。在输出的最外层，你会看到由 4 个大括号包裹的“数据段”。

T（中间层，每个大格子有 8 行）：
Time / Sequence Length = 8（代表每个句子有 8 个字符）：每一个大格子里，整整齐齐地排了 8 行 数字（比如第一组里从 [0.1808, -0.0700] 一直排到 [-0.9211, 1.5433]）

C（最内层，每行有 2 个实数，如 [0.1808, -0.0700]）：
嵌入维度（Embedding Dimension）/ 特征数
C


----------------------------------------------------------------------------------

用矩阵乘法代替for循环

In [ ]:
import torch

# 为了对比，固定随机种子
torch.manual_seed(42)

# 1. 创建一个 3x3 的全一矩阵 a
a = torch.ones(3, 3)

# 2. 创建一个 3x2 的随机整数矩阵 b (范围在 0 到 9 之间)，并转为浮点型以便后续计算
b = torch.randint(0, 10, (3, 2)).float()

# 3. 计算：进行矩阵乘法（用 @ 符号）
# 形状对齐: [3, 3] @ [3, 2] ──> [3, 2]
c = a @ b

print('a (全一矩阵) =')
print(a)

print('--')
print('b (输入矩阵) =')
print(b)

print('--')
print('c (矩阵乘积结果) =')
# 观察 c 的值，它的每一行都是 b 的列累加之和
print(c)

a (全一矩阵) =
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
--
b (输入矩阵) =
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c (矩阵乘积结果) =
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


引入`tril`

In [ ]:
torch.tril(a)

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [ ]:
import torch

# 为了对比，固定随机种子
torch.manual_seed(42)

# 1. 创建一个 3x3 的全一矩阵 a
a = torch.tril(torch.ones(3, 3))

# 2. 创建一个 3x2 的随机整数矩阵 b (范围在 0 到 9 之间)，并转为浮点型以便后续计算
b = torch.randint(0, 10, (3, 2)).float()

# 3. 计算：进行矩阵乘法（用 @ 符号）
# 形状对齐: [3, 3] @ [3, 2] ──> [3, 2]
c = a @ b

print('a (全一矩阵) =')
print(a)

print('--')
print('b (输入矩阵) =')
print(b)

print('--')
print('c (矩阵乘积结果) =')
# 观察 c 的值，它的每一行都是 b 的列累加之和！ [3]
print(c)

a (全一矩阵) =
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
--
b (输入矩阵) =
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c (矩阵乘积结果) =
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


In [ ]:
# @title
# 神经网络前向传播：因果遮蔽（Causal Masking）底层矩阵计算交互
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body { font-family: 'Noto Sans SC', sans-serif; }
    .matrix-cell { transition: all 0.3s ease; }
  </style>
</head>
<body class="bg-slate-950 text-slate-100 flex items-center justify-center p-4">
  <div class="w-full max-w-2xl bg-slate-900 border border-slate-800 rounded-3xl p-6 shadow-2xl">
    <h2 class="text-xl font-bold text-center text-cyan-400 mb-1">因果遮蔽（Causal Masking）矩阵乘法解析</h2>
    <p class="text-xs text-slate-400 text-center mb-6">点击下方时间步，观察矩阵“行乘列”如何实现屏蔽未来、只加过去</p>

    <!-- 1. 时间步切换按钮 (Time Steps) -->
    <div class="flex gap-2 p-1 bg-slate-950 rounded-xl mb-6">
      <button id="btn-t0" onclick="showStep(0)" class="flex-1 py-2 px-3 text-xs font-semibold rounded-lg bg-cyan-600 text-white focus:outline-none transition-all duration-300">
        📅 时间步 t = 0 (看第1个字)
      </button>
      <button id="btn-t1" onclick="showStep(1)" class="flex-1 py-2 px-3 text-xs font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300">
        📅 时间步 t = 1 (看前2个字)
      </button>
      <button id="btn-t2" onclick="showStep(2)" class="flex-1 py-2 px-3 text-xs font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300">
        📅 时间步 t = 2 (看前3个字)
      </button>
    </div>

    <!-- 2. 矩阵可视化对齐区域 (Matrix Visualization) -->
    <div class="flex flex-col md:flex-row items-center justify-center gap-4 mb-6 font-mono text-sm">

      <!-- 矩阵 A (下三角归一化) -->
      <div class="flex flex-col items-center">
        <span class="text-xs text-slate-400 mb-2 font-sans font-bold">下三角归一化权重 A</span>
        <div class="grid grid-cols-3 gap-1 p-2 bg-slate-950 rounded-xl border border-slate-800">
          <div id="a-0-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">1.0</div>
          <div id="a-0-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-600">0.0</div>
          <div id="a-0-2" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-600">0.0</div>

          <div id="a-1-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">0.5</div>
          <div id="a-1-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">0.5</div>
          <div id="a-1-2" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-600">0.0</div>

          <div id="a-2-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">0.33</div>
          <div id="a-2-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">0.33</div>
          <div id="a-2-2" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">0.33</div>
        </div>
      </div>

      <span class="text-xl font-bold text-slate-500 my-auto">@</span>

      <!-- 矩阵 B (输入特征) -->
      <div class="flex flex-col items-center">
        <span class="text-xs text-slate-400 mb-2 font-sans font-bold">输入特征 B (3x2)</span>
        <div class="grid grid-cols-2 gap-1 p-2 bg-slate-950 rounded-xl border border-slate-800">
          <div id="b-0-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">2.0</div>
          <div id="b-0-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">7.0</div>

          <div id="b-1-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">6.0</div>
          <div id="b-1-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">4.0</div>

          <div id="b-2-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">6.0</div>
          <div id="b-2-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">5.0</div>
        </div>
      </div>

      <span class="text-xl font-bold text-slate-500 my-auto">=</span>

      <!-- 结果矩阵 C (输出均值) -->
      <div class="flex flex-col items-center">
        <span class="text-xs text-slate-400 mb-2 font-sans font-bold">结果均值 C (3x2)</span>
        <div class="grid grid-cols-2 gap-1 p-2 bg-slate-950 rounded-xl border border-slate-800">
          <div id="c-0-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">2.0</div>
          <div id="c-0-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">7.0</div>

          <div id="c-1-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">4.0</div>
          <div id="c-1-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">5.5</div>

          <div id="c-2-0" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">4.67</div>
          <div id="c-2-1" class="w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 bg-slate-900">5.33</div>
        </div>
      </div>
    </div>

    <!-- 3. 动态数学解释区域 (Math & Physical Explanation) -->
    <div class="p-4 bg-slate-950/50 rounded-2xl border border-slate-800 space-y-4">
      <div>
        <span class="text-xs font-bold text-cyan-400 uppercase tracking-wider block mb-1">数学代入计算</span>
        <div id="math-explanation" class="text-sm font-mono text-slate-300 leading-relaxed">
          <!-- 这里将被 JS 动态注入 -->
        </div>
      </div>
      <div>
        <span class="text-xs font-bold text-amber-400 uppercase tracking-wider block mb-1">因果遮蔽机制物理意义</span>
        <p id="physical-explanation" class="text-sm text-slate-300 leading-relaxed">
          <!-- 这里将被 JS 动态注入 -->
        </p>
      </div>
    </div>
  </div>

  <script>
    // 定义三步的配置参数
    const stepsConfig = {
      0: {
        math: `
          • C[0, 0] = (1.0 × 2.0) + (0.0 × 6.0) + (0.0 × 6.0) = <strong class="text-cyan-400">2.0</strong><br>
          • C[0, 1] = (1.0 × 7.0) + (0.0 × 4.0) + (0.0 × 5.0) = <strong class="text-cyan-400">7.0</strong>
        `,
        physics: `
          此时处于第 1 个时间步。网络在预测第 2 个字时，上文只有第一个字 <span class="bg-cyan-950 px-1 rounded text-cyan-400 font-bold">[2.0, 7.0]</span> [3]。<br>
          未来时间步的输入（第2、3个字）在 A 矩阵中<b>对应的权重全为 0.0</b>，这意味着它们的特征被完全<b>遮蔽（Masked）</b>，绝对无法向当前步骤泄露任何未来信息 [3]！
        `
      },
      1: {
        math: `
          • C[1, 0] = (0.5 × 2.0) + (0.5 × 6.0) + (0.0 × 6.0) = <strong class="text-cyan-400">4.0</strong><br>
          • C[1, 1] = (0.5 × 7.0) + (0.5 × 4.0) + (0.0 × 5.0) = <strong class="text-cyan-400">5.5</strong>
        `,
        physics: `
          此时处于第 2 个时间步。网络在预测第 3 个字时，上文有前 2 个字。我们对前 2 个字的特征进行<b>等权重求均值（各占 0.5）</b> [3]：<br>
          <span class="font-mono text-xs bg-slate-950 p-1.5 rounded block my-1">([2.0, 7.0] + [6.0, 4.0]) / 2 = [4.0, 5.5]</span>
          而处于未来的第 3 个字对应的权重为 <b>0.0</b>，依然被稳稳地阻挡在未来之外，确保了因果律 [3]。
        `
      },
      2: {
        math: `
          • C[2, 0] = (0.33 × 2.0) + (0.33 × 6.0) + (0.33 × 6.0) = <strong class="text-cyan-400">4.67</strong><br>
          • C[2, 1] = (0.33 × 7.0) + (0.33 × 4.0) + (0.33 × 5.0) = <strong class="text-cyan-400">5.33</strong>
        `,
        physics: `
          此时处于第 3 个时间步。网络在预测第 4 个字时，能够看到前面全部 3 个字。我们将这 3 个字在二维地图上的特征坐标全部融合，<b>各占 0.33 的权重求出均值</b> [3]：<br>
          <span class="font-mono text-xs bg-slate-950 p-1.5 rounded block my-1">([2.0, 7.0] + [6.0, 4.0] + [6.0, 5.0]) / 3 = [4.67, 5.33]</span>
          此时，3 个字的历史信息已经全部完美融合成为了当前位置的语义特征 [3]。
        `
      }
    };

    function clearHighlights() {
      // 清空 A 矩阵
      for (let r = 0; r < 3; r++) {
        for (let c = 0; r < 3; c++) {
          const el = document.getElementById(`a-${r}-${c}`);
          if (!el) break;
          el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-500 matrix-cell";
        }
      }
      // 清空 B 矩阵
      for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 2; c++) {
          const el = document.getElementById(`b-${r}-${c}`);
          if (!el) break;
          el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-400 matrix-cell";
        }
      }
      // 清空 C 矩阵
      for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 2; c++) {
          const el = document.getElementById(`c-${r}-${c}`);
          if (!el) break;
          el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-slate-800 text-slate-400 matrix-cell";
        }
      }
    }

    function showStep(t) {
      clearHighlights();

      // 1. 更新按钮激活状态
      for (let i = 0; i < 3; i++) {
        const btn = document.getElementById(`btn-t${i}`);
        if (i === t) {
          btn.className = "flex-1 py-2 px-3 text-xs font-semibold rounded-lg bg-cyan-600 text-white focus:outline-none transition-all duration-300";
        } else {
          btn.className = "flex-1 py-2 px-3 text-xs font-semibold rounded-lg text-slate-400 hover:text-white focus:outline-none transition-all duration-300";
        }
      }

      // 2. 高亮当前行 A 矩阵中参与计算的格点
      for (let c = 0; c < 3; c++) {
        const el = document.getElementById(`a-${t}-${c}`);
        if (c <= t) {
          el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-cyan-500 bg-cyan-950/70 text-cyan-300 font-bold matrix-cell";
        } else {
          // 对应未来，为 0 的格子
          el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-dashed border-red-900/50 bg-red-950/20 text-red-700/60 font-medium matrix-cell";
        }
      }

      // 3. 高亮当前 B 矩阵中参与计算的行
      for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 2; c++) {
          const el = document.getElementById(`b-${r}-${c}`);
          if (r <= t) {
            el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-amber-600/80 bg-amber-950/40 text-amber-200 font-medium matrix-cell";
          } else {
            // 被遮蔽的未来
            el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-dashed border-slate-800/80 bg-slate-950 text-slate-700/40 matrix-cell";
          }
        }
      }

      // 4. 高亮 C 矩阵中输出的对应行
      for (let c = 0; c < 2; c++) {
        const el = document.getElementById(`c-${t}-${c}`);
        el.className = "w-12 h-9 flex items-center justify-center rounded-lg border border-emerald-500 bg-emerald-950/70 text-emerald-300 font-bold matrix-cell shadow-lg shadow-emerald-500/10";
      }

      // 5. 注入数学和物理说明
      document.getElementById('math-explanation').innerHTML = stepsConfig[t].math;
      document.getElementById('physical-explanation').innerHTML = stepsConfig[t].physics;
    }

    // 默认展示第一步
    showStep(0);
  </script>
</body>
</html>
"""

HTML(html_code)

应用

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
wei
#此处仍是加权平均
#keepdim=True 意味着“求和时保留原有的维度数（依然保持为 2D 矩阵）” 。此时，wei.sum(1, keepdim=True) 输出的形状为 [8, 1]（一个含有 1 列、8 行的列向量）

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

运算结果y

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x  # (T, T) @ (B, T, C) ----> (B, T, C)
xbow2

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [ ]:
xbow[0],xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

可以看到，矩阵乘法得到的结果与for循环一致

引入Causal Attention Masking的实现方法

在“版本二”中，我们通过手动算除法（`wei / wei.sum(...)`）来求平均值，但在真实的大模型中，**我们不能用这种除法，必须使用负无穷（`-inf`）配合 `Softmax` 来实现**



一、 为什么要引入 `-inf` 和 `Softmax`？

1. 真实大模型（Self-Attention）的运行状况
在真实的自注意力机制中，权重矩阵 `wei`（即注意力分数）是由字与字之间的“亲疏关系”动态计算出来的（即 Query 与 Key 的点积：$Q K^T$）。这些算出来的原始分数（Logits）可以是任何实数，比如：
$$\text{wei}_{\text{原始}} = \begin{bmatrix} 2.3 & -1.2 & 0.4 \\ 0.5 & 3.1 & -0.8 \\ -1.5 & 0.2 & 1.1 \end{bmatrix}$$

要对这些原始分数进行归一化，使其转为概率分布。在多分类和注意力机制中，标准的归一化工具就是 `Softmax`

---

 2. 直接对下三角矩阵用 `Softmax` 会怎样？
假设想把未来（右上角）的特征抹去。如果简单地把未来置为 `0`：
$$\text{wei}_{\text{直接置0}} = \begin{bmatrix} 2.3 & 0 & 0 \\ 0.5 & 3.1 & 0 \\ -1.5 & 0.2 & 1.1 \end{bmatrix}$$

当我们对这一行应用 `Softmax` 时，会发生计算错误：
* 根据对数定义，$e^0 = 1$
* 这意味着，**即使把未来的权重设为了 0，经过 Softmax 指数运算后，依然会分到 $1$ 的权重分**，即未来的信息依然会泄露进来，无法被遮蔽

---

 3. 解法：引入负无穷 $-\infty$（`float('-inf')`）
为了在 `Softmax` 运算下彻底将未来抹去，必须把未来的原始分数设为 **负无穷大（$-\infty$）**

* **数学原理**：
  根据指数函数的极限 ：
  $$\lim_{z \to -\infty} e^z = 0$$
* **结果**：当在未来通道填入 $-\infty$ 时，经过 `Softmax` 指数计算，$e^{-\infty}$ 会被**化为 0**，未来时刻的字符将分到绝对为 **$0\%$** 的注意力权重

---

 二、 代码底层

观察 **“Version 3: use Softmax”** 代码，看数学演变：

步骤 ①：制造下三角的 0/1 面板（`tril`）
```python
tril = torch.tril(torch.ones(T, T))
```
产生一个下半部分全为 1，右上角全为 0 的 $8 \times 8$ 面板

步骤 ②：用 `-inf` 填充未来（`masked_fill`）
```python
wei = torch.zeros((T,T)) # 假设这是 QK^T 计算出来的原始得分（这里用全 0 代替）
wei = wei.masked_fill(tril == 0, float('-inf'))
```
* **`tril == 0`**：找出右上角（未来时刻）的所有格子
* **`masked_fill(..., float('-inf'))`**：把这些格子的数值强行**替换为负无穷大 `-inf`**
* **结果**：得到矩阵 ：
  $$\text{wei} = \begin{bmatrix} 0.0 & -\infty & -\infty & \dots & -\infty \\ 0.0 & 0.0 & -\infty & \dots & -\infty \\ 0.0 & 0.0 & 0.0 & \dots & -\infty \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 0.0 & 0.0 & 0.0 & \dots & 0.0 \end{bmatrix}_{8 \times 8}$$

步骤 ③：运行 `Softmax` 归一化
```python
wei = F.softmax(wei, dim=-1)
```
对 `wei` 的每一行横向运行一次 `Softmax` 。以第 1 行为例 `[0.0, 0.0, -inf, -inf ...]`：
* **计算指数**：$e^0 = 1$，$e^0 = 1$，$e^{-\infty} = 0$，$e^{-\infty} = 0 \dots$
* **求和**：$1 + 1 + 0 + 0 \dots = 2$
* **归一化概率**：
  * 第 0 个位置：$\frac{1}{2} = 0.5$
  * 第 1 个位置：$\frac{1}{2} = 0.5$
  * 后面的未来位置：$\frac{0}{2} = 0.0$
* **结果**：这一行变为了 `[0.5, 0.5, 0.0, 0.0 ...]`

整个 `wei` 矩阵在经历 `Softmax` 之后，**变回了在版本二中看到的下三角均值矩阵**

---

三、 因果自注意力机制（Causal Self-Attention）

代码**契合了人类在语言预测时的真实规律** ：

1. **普适性**：
   未来，我们把 `wei = torch.zeros((T, T))` 替换为通过注意力计算出来的有大有小的实数值 `wei = q @ k.transpose(-2, -1)`
2. **兼容性**：
   无论实数值怎么变（比如算出来是 `[2.5, -1.8, 4.3]`），运行 `masked_fill(tril == 0, float('-inf'))`，就能**一键把所有未来的格子全部置为 $-\infty$**
3. **因果对齐**：
   最后，送入 `F.softmax(wei, dim=-1)`，它会自动根据权重大小对“过去”的历史进行最合理的比例分配（比如 `[0.85, 0.15, 0.00]`），而**处于未来的字符则永远分到绝对为 0 的权重**，实现了绝对的因果律控制

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

使用softmax函数

In [ ]:
import torch.nn.functional as F

# 创建下三角矩阵：1表示"可以看"，0表示"不能看"（未来）
tril = torch.tril(torch.ones(T, T))

# 创建一个全0的权重矩阵，形状(T,T)
wei = torch.zeros((T, T))

# 把tril中为0的位置（代表未来）填成负无穷
# 这样后面softmax后，这些位置的权重会变成0
wei = wei.masked_fill(tril == 0, float('-inf'))

# 对每一行做softmax，把得分变成"加和为1"的权重
# -inf经过softmax后会变成0
wei = F.softmax(wei, dim=-1)

# 用权重矩阵对x做加权求和
# wei(T,T) @ x(B,T,C) → xbow3(B,T,C)
xbow3 = wei @ x

# 检查这个矩阵方法和之前手写循环的方法结果是否一致
print(torch.allclose(xbow, xbow3))
# False ← 因为浮点数精度误差，但数学上是等价的

False


In [ ]:
xbow[0],xbow3[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

单头注意力实现精简版自注意力机制

In [ ]:
# 1. 固定的随机数种子，确保每次运行结果一致，便于调试对齐
torch.manual_seed(1337)

# 2. 超参数配置：
# B = 4  (Batch size，一次并行处理 4 个独立的句子)
# T = 8  (Time/Sequence Length，每个句子最长有 8 个字)
# C = 32 (Channel/Embedding Dimension，每个字被编码为一个 32 维的语义坐标向量)
B, T, C = 4, 8, 32

# 3. 随机生成输入数据 x，形状为 [4, 8, 32]
# 代表当前输入了 4 个句子，每句 8 个字，每个字拥有一个 32 维的语义向量
x = torch.randn(B, T, C)

# 4. 制造一个 8x8 的下三角全一矩阵面板
tril = torch.tril(torch.ones(T, T))

# 5. 制造一个 8x8 的全零初始注意力分数矩阵 wei
wei = torch.zeros(T, T)

# 6.因果遮蔽 (Causal Masking)
# 寻找 tril 矩阵中为 0 的格子（即代表未来的右上角部分）
# 用 .masked_fill_() 将这些未来的格子强行替换为负无穷大 -inf
wei = wei.masked_fill(tril == 0, float('-inf'))

# 7.对最后一维进行 Softmax 归一化
# 由于指数运算中 e^(-inf) = 0，未来的所有权重在这一步被完美地化为了 0.0
# 而历史部分的权重被自动、均等地分配为了 1.0, 0.5 或 0.33 等
wei = F.softmax(wei, dim=-1)

# 8.一键矩阵相乘
# 形状对齐：wei [4, 8, 8] (通过广播自动复制)  @  x [4, 8, 32]  ──>  out [4, 8, 32]
# 这一步在底层由 GPU 并行运行，瞬间算完了 4 个句子、每个句子 8 个字各自的历史特征均值
out = wei @ x

# 9. 打印输出数据的形状 ──> 预期输出: torch.Size([4, 8, 32])
print(out.shape)

torch.Size([4, 8, 32])


通过 Query（查询） 和 Key（键） 的数学交互，实现让字与字之间进行“主动对话”和“关系动态打分”的自注意力机制

note：


在前面的“版本三”中，注释掉的那行 `# wei = torch.zeros((T,T))` 是一个所有历史权重一律均分的常数矩阵

而新增加的这几行代码，通过 **Query（查询）** 和 **Key（键）** 交互，**实现自注意力机制**


单头自注意力机制（Self-Attention）

---

直觉：什么是 Query 与 Key？

在之前的均值模型中，历史字符分到的注意力都是均等的（如各占 $0.5$ 或 $0.33$）。但真实的语言是有侧重点的。为了让模型学会“选择性倾听”，需要给每一个字符分配了两个**向量角色**：

1. **Query（查询向量，$\vec{q}$）**：
   * **含义**：*“我当前处于位置 $i$，正在寻找什么样的上文特征？”*
2. **Key（键向量，$\vec{k}$）**：
   * **含义**：*“我是处于历史位置 $j$ 的词，我能提供什么样的特征？”*


如何发生化学反应？（点积打分）：

将第 $i$ 个词的 Query 与第 $j$ 个词的 Key 进行**点积（Dot Product）乘法**：
$$\text{关系得分 } \text{wei}_{i,j} = q_i \cdot k_j$$

* 如果第 $i$ 个词的“需求（Query）”与第 $j$ 个词的“条件（Key）”**高度匹配**，它们的点积分数就会是一个**极大的正数**（模型会施加巨大的注意力）
* 如果不匹配，分数就会很低或为负数（会选择性忽略它）

---

逐行代码与数学维度流转

新加的几行代码：

1. `head_size = 16`
* **意义**：注意力空间的维度。我们将每个字符原本 32 维的语义空间（$C=32$），投影（降维）到一个更精简的 **16 维注意力空间**（`head_size=16`）

2. 定义 Query 和 Key 投影矩阵
```python
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(in_features=C, out_features=head_size) # 逻辑等价
```
* **意义**：使用全连接线性层（不加偏置）作为特征转换器

3. 计算所有位置的 Query 和 Key（`self(idx)` 内部）
```python
logits = self.token_embedding_table(idx) # x 的形状为 (B, T, C) -> [4, 8, 2] (假设 C=2)
# 实际中 C=65，这里我们直接让它查表获取原始表示
```
在这一步，我们计算出整个 Batch 中，所有位置对应的 $\vec{k}$ 和 $\vec{q}$ 向量：
* `k = key(x)` $\longrightarrow$ 形状为 **`(B, T, head_size)`** ──> 即 `[4, 8, 16]`
* `q = query(x)` $\longrightarrow$ 形状为 **`(B, T, head_size)`** ──> 即 `[4, 8, 16]`

4. 运算：计算注意力得分矩阵 `wei`
```python
wei = q @ k.transpose(-2, -1)
```
这是一个维度对齐操作：
* **`k.transpose(-2, -1)`**：
  将 `k` 的最后两个维度（时间 $T$ 和特征 $head\_size$）进行**转置互换**。
  转置后的 `k` 形状变为了 **`(B, head_size, T)`** ──> 即 `[4, 16, 8]`

* **进行矩阵相乘（`@`）**：
  $$\text{Shape: } \underbrace{(B, \ T, \ 16)}_{\text{Query 矩阵 q}} \quad \times \quad \underbrace{(B, \ 16, \ T)}_{\text{转置后的 Key 矩阵}} \quad \longrightarrow \quad \underbrace{(B, \ T, \ T)}_{\text{动态注意力得分矩阵 wei}}$$
  * **意义**：输出的 `wei` 是一个 **`[4, 8, 8]`** 的 3D 张量。对于这 4 个句子中的每一个，我们都算出了一个 $8 \times 8$ 的矩阵。
  * 矩阵中的每一个格点 $(i, j)$，正是第 $i$ 个词的 Query 和第 $j$ 个词的 Key 进行点积算出来的**亲疏关系得分**

可视化理解

In [ ]:
# @title
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Output
import ipywidgets as widgets

# 1. 严格复现 Karpathy 截图中的参数与初始化
torch.manual_seed(1337)
B, T, C = 4, 8, 32  # batch, time, channels
x = torch.randn(B, T, C)

head_size = 16
# 模拟单头自注意力的线性映射（不带偏置）
key_layer = torch.nn.Linear(C, head_size, bias=False)
query_layer = torch.nn.Linear(C, head_size, bias=False)

# 前向计算得到 q 和 k
k = key_layer(x)  # (B, T, 16)
q = query_layer(x)  # (B, T, 16)

# 计算原始注意力得分矩阵 wei: (B, T, 16) @ (B, 16, T) -> (B, T, T)
wei_raw = q @ k.transpose(-2, -1)

# 创建下三角掩码 tril
tril = torch.tril(torch.ones(T, T))

# 注入 -inf 掩码
wei_masked = wei_raw.masked_fill(tril == 0, float('-inf'))

# 经过 Softmax 归一化
wei_softmax = F.softmax(wei_masked, dim=-1)

# 2. 构建 Colab 可交互面板
def plot_matrix_step(batch_idx, step):
    fig, ax = plt.subplots(1, 1, figsize=(7, 6))

    # 提取当前 Batch 的矩阵
    raw_mat = wei_raw[batch_idx].detach().numpy()
    masked_mat = wei_masked[batch_idx].detach().numpy()
    softmax_mat = wei_softmax[batch_idx].detach().numpy()

    # 模拟文本标签（方便理解自回归）
    words = [f"Token_{i}" for i in range(T)]

    if step == 0:
        # 步骤 0: Q @ K^T 原始碰撞得分
        im = ax.imshow(raw_mat, cmap="coolwarm")
        ax.set_title(f"Step 1: Q @ K^T Raw Attention Scores (Batch {batch_idx})\n[All tokens can see each other]", fontsize=11, pad=15)
        mat_data = raw_mat
        fmt = ".2f"
    elif step == 1:
        # 步骤 1: 注入下三角遮掩 (Masking)
        im = ax.imshow(masked_mat, cmap="coolwarm", vmin=-5, vmax=5)
        ax.set_title(f"Step 2: After Masked Fill (tril == 0) with -inf\n[Future tokens are blocked!]", fontsize=11, pad=15)
        mat_data = masked_mat
        fmt = ".2f"
    else:
        # 步骤 2: Softmax 归一化后的最终权重
        im = ax.imshow(softmax_mat, cmap="YlGnBu", vmin=0, vmax=1)
        ax.set_title(f"Step 3: After Softmax = Final Attention Weights\n[Perfect Causal Lower-Triangle Matrix!]", fontsize=11, pad=15)
        mat_data = softmax_mat
        fmt = ".2f"

    # 在格子内部绘制具体数值
    for i in range(T):
        for j in range(T):
            val = mat_data[i, j]
            if val == float('-inf') or val < -100:
                text_str = "-inf"
                color = "white" if step == 1 else "black"
            else:
                text_str = f"{val:.2f}"
                color = "white" if np.abs(val) > (mat_data.max()*0.6) and step != 2 else "black"
            ax.text(j, i, text_str, ha="center", va="center", color=color, fontweight='bold', fontsize=9)

    # 漂亮的轴线标签配置
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels(words, rotation=45, ha="right")
    ax.set_yticklabels(words)
    ax.set_xlabel("Keys (Looking Back into History)", fontsize=10, labelpad=10)
    ax.set_ylabel("Queries (Current Tokens)", fontsize=10, labelpad=10)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

# 创建滑块控件
batch_slider = IntSlider(min=0, max=B-1, step=1, value=0, description='Batch 序列:')
step_slider = IntSlider(min=0, max=2, step=1, value=0, description='计算步骤:')

# 自自定义步骤滑块的文字说明，方便肉眼 debug
step_labels = widgets.HTML(
    value="<p><b>计算步骤指南:</b><br>"
          "0: 原始 <b>q @ k.transpose</b> 碰撞（右上角有数字，代表信息泄露，未来会剧透给过去）<br>"
          "1: 运行 <b>masked_fill</b>（右上角未来的剧透全部被无情替换成了 <b>-inf</b>）<br>"
          "2: 运行 <b>F.softmax</b>（因为 e^{-inf} = 0，右上角全部归零，每行加起来等于1，完美自回归！）</p>"
)

# 渲染控制面板
ui = widgets.VBox([widgets.HBox([batch_slider, step_slider]), step_labels])
out = widgets.interactive_output(plot_matrix_step, {'batch_idx': batch_slider, 'step': step_slider})
display(ui, out)


Output()

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

B, T, C = 4, 8, 32 # 4个样本，8个时间步，每个字32维
x = torch.randn(B, T, C)

# 1. 核心创新：引入单头自注意力机制 (Single Head Self-Attention)
head_size = 16  # 定义注意力特征空间的大小为 16

# 定义 Key 和 Query 的线性投影变换矩阵（不带偏置）
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)

# 计算每个字符对应的 Key 向量，形状变为: (B, T, head_size) -> [4, 8, 16]
k = key(x)
# 计算每个字符对应的 Query 向量，形状变为: (B, T, head_size) -> [4, 8, 16]
q = query(x)

# 计算字与字之间的动态亲疏得分
# k.transpose(-2, -1) 将 k 形状转置为: (B, 16, T) -> [4, 16, 8] [3]
# 矩阵相乘：(B, T, 16) @ (B, 16, T) ──> 得到动态的 wei 矩阵 (B, T, T) -> [4, 8, 8]
wei = q @ k.transpose(-2, -1)

# 2. 经典的因果遮蔽与 Softmax 归一化（与上一节相同）
tril = torch.tril(torch.ones(T, T))

# 即使 q · k 算出来的未来分数再高，只要 tril == 0（代表未来），直接一键用 -inf 强行遮蔽覆盖掉
wei = wei.masked_fill(tril == 0, float('-inf'))

# 通过 Softmax 归一化。此时：
# 未来的格子分配到绝对为 0.0 的注意力；
# 过去的格子根据刚才 q·k 算出的得分，动态、有大有小地合理分配注意力权重
wei = F.softmax(wei, dim=-1)

# 最终完成特征融合：让这一组动态变化的权重 wei 去加权融合原始特征 x
out = wei @ x

# 打印最终输出形状，依然保持为: torch.Size([4, 8, 32])
print(out.shape)

torch.Size([4, 8, 32])


In [ ]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089

In [ ]:
q.shape

torch.Size([4, 8, 16])

In [ ]:
k.shape

torch.Size([4, 8, 16])

引入value

In [ ]:
# 1. 声明并初始化 Q, K, V
head_size = 16
key = nn.Linear(C, head_size, bias=False)   # 提问标准转换器
query = nn.Linear(C, head_size, bias=False) # 自身条件转换器
value = nn.Linear(C, head_size, bias=False) # 语义特征转换器

# 2. 将原始输入映射到各自的特征空间中
k = key(x)      # 形状: (B, T, 16)
q = query(x)    # 形状: (B, T, 16)
v = value(x)    # 新增：形状: (B, T, 16)

# 3. 计算字与字之间的动态亲疏得分
# 矩阵相乘：(B, T, 16) @ (B, 16, T) ──> 得到动态得分 wei (B, T, T)
wei = q @ k.transpose(-2, -1)

# 4. 因果遮蔽与归一化（不看未来，并分配合理的注意力百分比）
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

# 5. 特征融合
# 不再粗暴地去乘以多噪、沉重的原始输入 x，而是让注意力权重 wei，去加权融合每个字精细化的特征v
# 矩阵相乘：(B, T, T) @ (B, T, 16) ──> 最终输出 out (B, T, 16)
out = wei @ v

# 6. 打印输出形状 ──> 预期输出: torch.Size([4, 8, 16])
print(out.shape)

torch.Size([4, 8, 16])


引入缩放

In [ ]:
# 自注意力机制中的缩放因子（Scaled Dot-Product）与 Softmax 饱和

import torch
import torch.nn.functional as F

# 假定 Batch_size = 4, Time = 8, head_size = 16
B, T, head_size = 4, 8, 16

# 1. 模拟生成服从标准正态分布（均值为 0，方差为 1）的 Query 和 Key 向量
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

# 2.计算：计算点积注意力得分，并乘以缩放因子 (head_size ** -0.5)
# head_size ** -0.5 等价于 1 / sqrt(head_size) = 1 / 4
wei = q @ k.transpose(-2, -1) * (head_size ** -0.5)

# 3. 运行方差（Variance）验证
# k 和 q 的原始方差均在 1.0 左右（对应 [153] 单元格输出：0.9006 与 1.0037）
print("k 的方差:", k.var().item())
print("q 的方差:", q.var().item())

# 经过缩放因子的调控，相乘后的注意力得分 wei 的方差被控制在了 1.0 左右
# 对应 单元格输出：tensor(0.9957)
print("wei 的方差:", wei.var().item())


# 4. 验证为什么高方差是 Softmax 的雷区
# 设定一组温和的原始 Logits 分数（方差小）
logits_small = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])

# 场景 A（有缩放）：直接输入分数进行 Softmax
probs_scaled = F.softmax(logits_small, dim=-1)
# 预期输出: [0.1925, 0.1426, 0.2351, 0.1426, 0.2872]
# 各个通道都有合理的机会分到权重，梯度流动正常
print("\n【有缩放】Softmax 输出:", probs_scaled)

# 场景 B（无缩放）：假设由于未缩放，数值被放大了 8 倍（方差极大）
logits_large = logits_small * 8  # 放大 8 倍后为: [0.8, -1.6, 2.4, -1.6, 4.0]
probs_unscaled = F.softmax(logits_large, dim=-1)
# 预期输出: [0.0326, 0.0030, 0.1615, 0.0030, 0.8000]
# 最大的 0.5 放大后占了 80% 的概率，其余通道全被压制为近乎 0.0
print("【无缩放】Softmax 输出:", probs_unscaled)

k 的方差: 0.9005609154701233
q 的方差: 1.003690242767334
wei 的方差: 0.9957307577133179

【有缩放】Softmax 输出: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
【无缩放】Softmax 输出: tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


可视化

In [ ]:
# @title
from IPython.display import HTML

html_code = """
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.tailwindcss.com"></script>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;500;700&display=swap');
    body { font-family: 'Noto Sans SC', sans-serif; }
    .bar { transition: height 0.4s cubic-bezier(0.4, 0, 0.2, 1); }
  </style>
</head>
<body class="bg-slate-950 text-slate-100 flex items-center justify-center p-4">
  <div class="w-full max-w-xl bg-slate-900 border border-slate-800 rounded-3xl p-6 shadow-2xl">
    <h2 class="text-lg font-bold text-center text-cyan-400 mb-2">Softmax 饱和度与梯度健康度演示</h2>
    <p class="text-xs text-slate-400 text-center mb-6">滑动调整“方差放大倍数”，观察 Softmax 输出和梯度的状态</p>

    <!-- 1. 滑块控制区 -->
    <div class="bg-slate-950 p-4 rounded-2xl border border-slate-800/80 mb-6 space-y-3">
      <div class="flex justify-between text-xs font-semibold">
        <span class="text-slate-400">方差乘数 (Multiplier)</span>
        <span id="multiplier-val" class="text-cyan-400 text-sm">1.0 x (已缩放健康状态)</span>
      </div>
      <input id="slider" type="range" min="1" max="10" step="0.5" value="1" oninput="updateViz(this.value)" class="w-full h-1.5 bg-slate-800 rounded-lg appearance-none cursor-pointer accent-cyan-500">
      <div class="flex justify-between text-[10px] text-slate-500">
        <span>1.0x (有缩放/健康)</span>
        <span>10.0x (无缩放/崩溃)</span>
      </div>
    </div>

    <!-- 2. 柱状图展示区 -->
    <div class="flex justify-around items-end h-32 bg-slate-950/40 border border-slate-800/80 rounded-2xl p-4 mb-6 relative">
      <div class="absolute top-2 left-4 text-[10px] text-slate-500 font-sans">SOFTMAX 输出概率分布</div>

      <!-- Bar 0 -->
      <div class="flex flex-col items-center w-12">
        <div id="bar-0" class="bar w-6 bg-cyan-500 rounded-t-md" style="height: 19.25%"></div>
        <span class="text-[10px] text-slate-400 mt-2 font-mono">0.1</span>
        <span id="val-0" class="text-[9px] text-cyan-400 font-mono mt-0.5">19.3%</span>
      </div>
      <!-- Bar 1 -->
      <div class="flex flex-col items-center w-12">
        <div id="bar-1" class="bar w-6 bg-cyan-500 rounded-t-md" style="height: 14.26%"></div>
        <span class="text-[10px] text-slate-400 mt-2 font-mono">-0.2</span>
        <span id="val-1" class="text-[9px] text-cyan-400 font-mono mt-0.5">14.3%</span>
      </div>
      <!-- Bar 2 -->
      <div class="flex flex-col items-center w-12">
        <div id="bar-2" class="bar w-6 bg-cyan-500 rounded-t-md" style="height: 23.51%"></div>
        <span class="text-[10px] text-slate-400 mt-2 font-mono">0.3</span>
        <span id="val-2" class="text-[9px] text-cyan-400 font-mono mt-0.5">23.5%</span>
      </div>
      <!-- Bar 3 -->
      <div class="flex flex-col items-center w-12">
        <div id="bar-3" class="bar w-6 bg-cyan-500 rounded-t-md" style="height: 14.26%"></div>
        <span class="text-[10px] text-slate-400 mt-2 font-mono">-0.2</span>
        <span id="val-3" class="text-[9px] text-cyan-400 font-mono mt-0.5">14.3%</span>
      </div>
      <!-- Bar 4 -->
      <div class="flex flex-col items-center w-12">
        <div id="bar-4" class="bar w-6 bg-cyan-500 rounded-t-md" style="height: 28.72%"></div>
        <span class="text-[10px] text-slate-400 mt-2 font-mono">0.5</span>
        <span id="val-4" class="text-[9px] text-cyan-400 font-mono mt-0.5">28.7%</span>
      </div>
    </div>

    <!-- 3. 诊断报告区 -->
    <div class="grid grid-cols-2 gap-4">
      <div class="p-3 bg-slate-950/50 rounded-2xl border border-slate-800">
        <span class="text-[10px] font-bold text-slate-500 block mb-1">梯度健康状态</span>
        <div class="flex items-center gap-2">
          <span id="status-light" class="w-3 h-3 rounded-full bg-emerald-500 animate-ping"></span>
          <span id="status-text" class="text-xs font-bold text-emerald-400">活跃 (Active)</span>
        </div>
      </div>
      <div class="p-3 bg-slate-950/50 rounded-2xl border border-slate-800">
        <span class="text-[10px] font-bold text-slate-500 block mb-1">输出状态诊断</span>
        <span id="diag-text" class="text-xs text-slate-300 block">百家争鸣，特征融合效果极好</span>
      </div>
    </div>
  </div>

  <script>
    const originalLogits = [0.1, -0.2, 0.3, -0.2, 0.5];

    function softmax(logits) {
      const exps = logits.map(x => Math.exp(x));
      const sumExps = exps.reduce((a, b) => a + b, 0);
      return exps.map(x => x / sumExps);
    }

    function updateViz(multiplier) {
      const m = parseFloat(multiplier);

      // 1. 计算缩放/放大的 logits
      const scaledLogits = originalLogits.map(x => x * m);
      const probs = softmax(scaledLogits);

      // 2. 更新滑块标签说明
      const label = document.getElementById('multiplier-val');
      if (m === 1.0) {
        label.className = "text-emerald-400 text-sm font-bold";
        label.innerHTML = "1.0 x (有缩放：完美保留方差)";
      } else if (m < 5.0) {
        label.className = "text-cyan-400 text-sm";
        label.innerHTML = `${m.toFixed(1)} x (未缩放：方差开始扩散)`;
      } else {
        label.className = "text-red-500 text-sm font-bold animate-pulse";
        label.innerHTML = `${m.toFixed(1)} x (无缩放：极度危险的溢出区)`;
      }

      // 3. 更新柱状图高度与文本
      for (let i = 0; i < 5; i++) {
        const bar = document.getElementById(`bar-${i}`);
        const textVal = document.getElementById(`val-${i}`);
        const percentage = probs[i] * 100;

        bar.style.height = `${percentage}%`;
        textVal.innerHTML = `${percentage.toFixed(1)}%`;

        // 依据高度改变颜色，凸显垄断效应
        if (percentage > 70) {
          bar.className = "bar w-6 bg-red-600 rounded-t-md";
          textVal.className = "text-[9px] text-red-500 font-bold font-mono mt-0.5";
        } else if (percentage < 3) {
          bar.className = "bar w-6 bg-slate-800 rounded-t-md";
          textVal.className = "text-[9px] text-slate-600 font-mono mt-0.5";
        } else {
          bar.className = "bar w-6 bg-cyan-500 rounded-t-md";
          textVal.className = "text-[9px] text-cyan-400 font-mono mt-0.5";
        }
      }

      // 4. 更新诊断诊断
      const statusLight = document.getElementById('status-light');
      const statusText = document.getElementById('status-text');
      const diagText = document.getElementById('diag-text');

      if (probs.some(p => p > 0.99) && m > 2) { // 某个概率接近1且乘数较大
        statusLight.className = "w-3 h-3 rounded-full bg-red-500 animate-pulse";
        statusText.className = "text-xs font-bold text-red-400";
        statusText.innerHTML = "饱和 (Saturated)";
        diagText.innerHTML = "注意力集中在少数几个词，梯度容易消失，模型学习效率低";
      } else if (probs.some(p => p < 0.01) && m > 2) { // 某个概率接近0且乘数较大
        statusLight.className = "w-3 h-3 rounded-full bg-red-500 animate-pulse";
        statusText.className = "text-xs font-bold text-red-400";
        statusText.innerHTML = "饱和 (Saturated)";
        diagText.innerHTML = "部分词权重接近零，无法接收梯度更新，模型表达力受限";
      } else if (m > 5) {
        statusLight.className = "w-3 h-3 rounded-full bg-yellow-500 animate-pulse";
        statusText.className = "text-xs font-bold text-yellow-400";
        statusText.innerHTML = "不稳定 (Unstable)";
        diagText.innerHTML = "方差过大，Softmax输出极端，梯度不稳定，可能导致NaN";
      } else {
        statusLight.className = "w-3 h-3 rounded-full bg-emerald-500 animate-ping";
        statusText.className = "text-xs font-bold text-emerald-400";
        statusText.innerHTML = "活跃 (Active)";
        diagText.innerHTML = "百家争鸣，特征融合效果极好";
      }
    }

    // 默认展示第一步
    updateViz(document.getElementById('slider').value);
  </script>
</body>
</html>
"""

HTML(html_code)

完整model代码

note

以下代码是标准的、与 OpenAI 官方完全等价的 **GPT-2 语言模型** 的单文件 PyTorch 实现。它不仅包含了自注意力机制，还涵盖了层归一化（LayerNorm）、多层感知机（MLP）、残差连接以及高阶的自回归文本生成器

在深入代码之前，先理清 **GPT-2 架构的 3 个核心设计**：


 1. Pre-LN（前置层归一化）残差控制


在早期的 Transformer（如 Transformer-Selt）中，LayerNorm 是挂在残差相加之后的（Post-LN）。但 GPT-2 将其改为了 **Pre-LN**：
* **公式数学**：
  $$x \leftarrow x + \text{Attention}(\text{LN}(x))$$
  $$x \leftarrow x + \text{MLP}(\text{LN}(x))$$

  
* **意义**：在数据进入注意力层和 MLP 之前先进行归一化（LN）。这极大地稳定了网络的梯度流，使得深层模型在开局训练时不易发生梯度爆炸或消失，是千亿参数大模型得以训练稳定的基石

2. 权重共享（Weight Tying / Embedding Sharing）
* **机制**：输入端的词嵌入层 `wte`（将单词 ID 映射为稠密向量，形状为 `[vocab_size, n_embd]`）与输出端的语言模型预测头 `lm_head`（将特征映射回词表得分，形状为 `[n_embd, vocab_size]`）**共享同一个权重矩阵（W 的转置）**
* **意义**：可以节省近三分之一的参数量，还能约束模型的表征空间，使得输入语义和输出预测在数学上完全互逆对齐

3. Flash Attention（闪电注意力机制）
* **原理**：标准的自注意力机制在计算 $Q K^T$ 关系矩阵时，会在显存中产生一个 $O(T^2)$ 级别的巨大中间矩阵，极易引发显存溢出（OOM）
* **机制**：PyTorch 2.0+ 推出了底层 CUDA 级高度融合的 `scaled_dot_product_attention`。它通过在线增量计算，**完全避免了在显存中存储高维注意力矩阵**，从而实现提速并极大节省显存

In [ ]:
"""
完整的 GPT 语言模型定义，全部浓缩在这一个单文件中。
参考资料：
1) OpenAI 官方 GPT-2 TensorFlow 实现：
https://github.com/openai/gpt-2/blob/master/src/model.py
2) HuggingFace/Transformers 官方的 PyTorch 实现：
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
"""

import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. 基础组件：LayerNorm (带可选偏置)
class LayerNorm(nn.Module):
    """
    自定义的层归一化，支持可选的偏置项 bias。
    因为标准 PyTorch 的 nn.LayerNorm 强制要求 bias=True，为了实现偏置开关，我们进行重写。
    """
    def __init__(self, ndim, bias):
        super().__init__()
        # 初始化权重为全 1，偏置为全 0（若配置中开启 bias）
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        # 实时调用底层的层归一化函数，ε（缩放平滑项）设为 1e-5
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

# 2. 核心组件：CausalSelfAttention (带因果掩码的多头自注意力机制)
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        # 确保 Embedding 特征维度可以被注意力头数整除
        assert config.n_embd % config.n_head == 0

        # 优化：
        # 我们不单独定义 q、k、v 三个 Linear 层。而是合并成一个 3 倍大小的 Linear。
        # 一次性算出所有头的 Q, K, V，随后通过 split 进行切分，极大提升矩阵计算并行度。
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)

        # 输出的投影变换层
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        # 正则化层
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        # 检测当前系统的 PyTorch 是否大于 2.0，若支持则一律采用极速的 Flash Attention
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("警告: 正在使用慢速注意力。Flash Attention 需要 PyTorch 2.0 及以上版本支持。")
            # 如果不支持 Flash Attention，则退回到传统的下三角因果掩码 (tril) 机制
            # 建立一个 [block_size, block_size] 的下三角矩阵并注册为 Buffer（不参与反向传播梯度更新）
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        # 获取输入张量的尺寸：B (批次大小), T (当前序列长度), C (特征维度，即 n_embd)
        B, T, C = x.size()

        # 1. 一键投射 Q, K, V：输入 x [B, T, C] ──> 投射为 3 倍大小 ──> 按维度 2 切分成三个 [B, T, C] 的张量
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)

        # 2. 维度流转（高维多头化）：
        # 以 k 为例，原本为 [B, T, C]，我们将其重塑为 4D: [B, T, n_head, head_size] （其中 head_size = C // n_head）
        # 接着调用 .transpose(1, 2) 交换维度 1 和维度 2，将注意力头提前到维度 1。
        # 最终形状变为: (B, nh, T, hs)，这一步是为了方便各头在后台完全并行进行矩阵相乘
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # [B, nh, T, hs]
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # [B, nh, T, hs]
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # [B, nh, T, hs]

        # 3. 运行注意力计算
        if self.flash:
            # 方案一（首选）：调用 CUDA 优化的 Flash Attention 内核，速度极快
            # is_causal=True 会自动在 GPU 寄存器级别上执行因果遮蔽（防止偷看未来）
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # 方案二（备用）：手动执行注意力计算
            # 矩阵相乘：[B, nh, T, hs] @ [B, nh, hs, T] ──> 得到注意力关系图 [B, nh, T, T]
            # 同时除以 sqrt(head_size) 的缩放因子，控制方差，防止 Softmax 饱和
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            # 利用下三角掩码，将右上角（未来时刻）的特征强制替换为 -inf
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            # 经过 Softmax 归一化，未来时刻分得 0% 的注意力
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            # 加权求和：[B, nh, T, T] @ [B, nh, T, hs] ──> 融合后的特征 [B, nh, T, hs]
            y = att @ v

        # 4. 多头输出拼接还原（维度流转回 3D）
        # 先将 nh 和 T 轴换回来 ──> 此时内存不连续，调用 .contiguous() 强行将其在内存中物理对齐
        # ──> 最后调用 .view 融合成 3D 形状: [B, T, C]
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # 输出线性投射
        y = self.resid_dropout(self.c_proj(y))
        return y

# 3. 基础组件：MLP (多层感知机 / 前馈传播网络)
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        # 第一层全连接：将特征维度上投影 4 倍 (如 768 ──> 3072)
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        # 激活函数：使用 GPT 标配的、比 ReLU 更平滑的 GELU 激活函数
        self.gelu    = nn.GELU()
        # 第二层全连接：将特征维度下投影回原始尺寸 (3072 ──> 768)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

# 4. 核心层块：Block (单个完整的 Transformer 解码器块)
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        # 第一个层归一化（自注意力前的 LN）
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        # 自注意力层
        self.attn = CausalSelfAttention(config)
        # 第二个层归一化（MLP 前的 LN）
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        # 前馈网络
        self.mlp = MLP(config)

    def forward(self, x):
        # 经典的 Pre-LN 结构与残差相加连接：
        # 输入先过 LN1 ──> 算自注意力 ──> 与输入 x 进行短路相加 (Residual Connection)
        x = x + self.attn(self.ln_1(x))
        # 接着过 LN2 ──> 算多层感知机 ──> 与 x 进行短路相加
        x = x + self.mlp(self.ln_2(x))
        return x

# 5. 配置参数类：GPTConfig
@dataclass
class GPTConfig:
    block_size: int = 1024         # 上下文最大长度
    vocab_size: int = 50304        # 词表大小。这里相比 GPT-2 默认的 50257 进行了向上取整（64的倍数），以迎合显卡硬件对齐提速
    n_layer: int = 12              # 12 个 Block 堆叠
    n_head: int = 12               # 12 个自注意力头
    n_embd: int = 768              # 768 维特征通道数
    dropout: float = 0.0           # Dropout 率
    bias: bool = True              # 是否使用偏置项

# 6. 父类网络：GPT 完全体
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        # 用 ModuleDict 建立字典式的子模型管理，便于清晰阅读和维护
        self.transformer = nn.ModuleDict(dict(
            # wte: Word Token Embedding (字符/词嵌入层)
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            # wpe: Word Position Embedding (绝对位置嵌入层)
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            # 堆叠 n_layer 层的 Block 解码器
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            # 最终的层归一化 (Final LayerNorm)
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        # 最终的语言模型输出投影头 (Language Modeling Head)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # 核心：权重共享（Weight Tying）
        # 将输入词嵌入 `wte` 的权重，直接共享绑定给最后的线性输出层 `lm_head`
        self.transformer.wte.weight = self.lm_head.weight

        # 递归初始化所有的网络层权重
        self.apply(self._init_weights)

        # 针对残差连接的最后一层线性投射进行了特殊的缩放初始化（防止层数加深后输出值发生膨胀，符合 GPT-2 论文规范）
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # 打印并汇报模型的总可学习参数量
        print("网络总参数量: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        返回模型的总参数量。
        如果 non_embedding=True（默认），则扣除位置编码矩阵 wpe 的参数量（因为它不随输入扩展，属于硬参数）。
        因为 wte 与最后的 lm_head 共享了权重，所以 wte 的参数仍会被计算在内。
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        """ 初始化权重的规则 """
        if isinstance(module, nn.Linear):
            # 线性层权重初始化为均值为 0，标准差为 0.02 的正态分布
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            # 嵌入层初始化为标准差为 0.02 的正态分布
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        # 确保传入的序列长度不超过我们在配置里设定的 block_size 上限
        assert t <= self.config.block_size, f"无法处理长度为 {t} 的输入，模型上限仅支持 {self.config.block_size}"

        # 生成对应当前序列长度的时间位置索引，例如 [0, 1, 2, ..., t-1]
        pos = torch.arange(0, t, dtype=torch.long, device=device) # 形状: [t]

        # 前向传播流程：
        tok_emb = self.transformer.wte(idx) # 词特征嵌入 ──> [b, t, n_embd]
        pos_emb = self.transformer.wpe(pos) # 位置特征嵌入 ──> [t, n_embd]

        # 将词嵌入和位置嵌入进行相加融合，并随机 Dropout 丢弃部分连接
        x = self.transformer.drop(tok_emb + pos_emb)

        # 顺序通过 12 层 Block 块
        for block in self.transformer.h:
            x = block(x)

        # 经过最后一层 LayerNorm
        x = self.transformer.ln_f(x)

        if targets is not None:
            # 场景 A：训练模式（输入了 Y_targets），我们需要计算 Loss
            logits = self.lm_head(x) # 形状: [b, t, vocab_size]
            # 计算交叉熵时，我们需要将 3D 的 logits 展平为 2D [b*t, vocab_size]，真实目标 Y 展平为 1D [b*t]
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # 场景 B：预测模式（推理/生成时，Y_targets 为 None）
            # 这是一个硬件计算优化点：
            # 在预测时，因为是自回归生成，我们只需要知道序列中【最后那一个字符】所预测出的下一个词
            # 因此，我们不需要去算前面 1 到 t-1 个字符的 lm_head 矩阵乘法，只取最后一列 [:, [-1], :] 来进行投影
            # 这能节省极大的矩阵计算开销和硬件显存占用。
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        """
        支持“模型剪裁”，动态缩减模型的最大上下文长度
        例如：我们加载了官方 1024 上下文的 GPT-2 预训练权重，但我们只想用 256 的长度进行小任务微调
        """
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        # 截断位置编码矩阵的大小，只保留前 block_size 行
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                # 截断自注意力里的 tril 缓冲面板大小
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        """
        一键从 Hugging Face 官方服务器下载并导入 OpenAI 原版的 GPT-2 预训练参数
        支持：'gpt2'（124M）、'gpt2-medium'（350M）、'gpt2-large'（774M）、'gpt2-xl'（1.5B）
        """
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {}
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("正在从 HuggingFace 官方下载/读取原版 GPT-2 权重: %s" % model_type)

        # 依据加载的模型类型，自动对齐网络层数、多头数和 Embedding 通道维度
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024),
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280),
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600),
        }[model_type]
        config_args['vocab_size'] = 50257 # 官方预训练一律使用原版 50257 大小的词表
        config_args['block_size'] = 1024  # 上下文上限一律为 1024
        config_args['bias'] = True        # 官方权重包含 bias 偏置
        if 'dropout' in override_args:
            print(f"将 Dropout 率强制重写为: {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']

        # 1. 先用当前的配置参数，实例化一个“空的（随机初始化）”本地模型
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # 过滤并扔掉非可更新参数的 tril 掩码

        # 2. 载入 Hugging Face 官方封装好的原版 GPT2 模型
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # 3. 参数对齐与转置（Weight Transpose）
        # 官方的权重文件中，其线性全连接层使用的是名为 "Conv1D" 的特殊模块，
        # 在 PyTorch 中，这需要我们将加载的参数矩阵进行【转置 t()】，然后才能安全地灌入我们的 nn.Linear 中
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        assert len(sd_keys_hf) == len(sd_keys), f"参数 Key 数量不对齐: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # 针对那些需要转置（从一维卷积到线性层）的特殊权重矩阵，先转置再拷贝
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # 正常的参数，直接进行一键无损复制
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        """
        高等级优化器配置规范（Weight Decay 分组过滤）：
        大模型训练的标配动作。我们不应该对网络里所有的参数一视同仁地施加正则化（Weight Decay）。
        1. 所有 2D 的权重矩阵（如 Linear 里的 w，Embedding 里的参数） -> 开启权重衰减
        2. 所有 1D 的偏置（bias）和归一化参数（LayerNorm 里的 scale 和 bias） -> 禁用权重衰减，避免欠拟合
        """
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}

        # 巧妙依据张量维度 dim() 过滤分组
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"参与 Weight Decay 惩罚的参数权重总数: {len(decay_params)}, 共 {num_decay_params:} 个参数")
        print(f"免除 Weight Decay 惩罚的参数权重总数: {len(nodecay_params)}, 共 {num_nodecay_params:} 个参数")

        # 检测当前硬件是否支持 PyTorch 的 fused AdamW（融合优化器）
        # 融合优化器可以将数十步更新小算子融合在 GPU 内运行，极大提速
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"正在启用 Fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ 评估当前的 MFU 硬件利用效率 """
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        flops_achieved = flops_per_iter * (1.0/dt)
        flops_promised = 312e12 # A100 GPU 理论半精度算力峰值为 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        大模型自回归文本生成器
        - temperature (采样温度): 控制生成文本的创造力与多样性
        - top_k: 是否开启 top_k 截断，限制模型只在概率最高的前 k 个候选词里挑选
        """
        for _ in range(max_new_tokens):
            # 1. 确保输入长度。如果历史上下文变得比 block_size 还要长，则对左侧历史执行截断
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]

            # 2. 前向传播：调用 forward 获得原始 logits 得分
            logits, _ = self(idx_cond)

            # 3. 提取最后一个时间步对应的预测 Logits，并除以温度系数进行缩放
            # - 温度极低 (如 0.2)：概率被拉大拉陡，模型生成结果变得高度固定、死板。
            # - 温度极高 (如 1.5)：概率平滑接近均等，模型极具创造力但可能胡言乱语。
            logits = logits[:, -1, :] / temperature

            # 4. 开启 Top-K 裁剪采样（剪掉概率极低、基本不通顺的胡言乱语词）
            if top_k is not None:
                # 提取概率最高的前 top_k 个词的分数，其余部分强制覆盖为 -inf（指数后概率为0）
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # 5. 用 Softmax 将 Logits 转化为 0~1 的概率
            probs = F.softmax(logits, dim=-1)

            # 6. 多项式随机概率抽样 (Biased Die)
            idx_next = torch.multinomial(probs, num_samples=1)

            # 7. 拼接：将新生成的词拼接回原序列的后面
            idx = torch.cat((idx, idx_next), dim=1)

        return idx